# 03 · Step verification with mmBERT (owner: Siddhi Karhekar)

Fine-tunes mmBERT (2025) as an entailment model (label 0 = entailment, 1 = neutral, 2 = contradiction)
on XNLI English plus IndicXNLI Hindi and Marathi (2022). The backend uses it to
(a) accept a Gemma 3 rephrasing only when it entails the verified step both ways and
(b) suggest agreement labels between guidelines offline; the situation owner confirms each one by hand.
Check the model and dataset ids on the Hugging Face Hub before running.

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn

In [ ]:
# Clone the repo (or upload it) and move into it.
!git clone https://github.com/<owner>/firstaid-assistant.git 2>/dev/null || true
%cd firstaid-assistant/backend
!python scripts/generate_queries.py

In [ ]:
from datasets import load_dataset, concatenate_datasets
parts = [load_dataset("xnli", "en", split="train[:60000]")]
for lang in ["hi", "mr"]:
    parts.append(load_dataset("Divyanshu/indicxnli", lang, split="train[:60000]"))
train = concatenate_datasets([p.select_columns(["premise", "hypothesis", "label"]) for p in parts]).shuffle(seed=1)
val = {lang: load_dataset("Divyanshu/indicxnli", lang, split="validation") for lang in ["hi", "mr"]}
val["en"] = load_dataset("xnli", "en", split="validation")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
name = "jhu-clsp/mmBERT-base"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=3)
enc = lambda b: tok(b["premise"], b["hypothesis"], truncation=True, max_length=160)
tr = train.map(enc, batched=True)
args = TrainingArguments("out_mmbert", per_device_train_batch_size=32, learning_rate=3e-5, num_train_epochs=2, fp16=True, logging_steps=200, save_strategy="no", report_to=[])
Trainer(model=model, args=args, train_dataset=tr, tokenizer=tok).train()

In [ ]:
for lang, ds in val.items():
    ds = ds.map(enc, batched=True)
    pred = Trainer(model=model, tokenizer=tok).predict(ds).predictions.argmax(-1)
    print(lang, "acc", round(accuracy_score(ds["label"], pred), 3), "macroF1", round(f1_score(ds["label"], pred, average="macro"), 3))
model.save_pretrained("../training/out/mmbert_verifier"); tok.save_pretrained("../training/out/mmbert_verifier")